# L1 — Single LLM Call

> **El bloque atómico de toda la IA generativa moderna.** Una llamada al modelo: le mandas texto, te devuelve texto.

## ¿Qué vas a aprender?

- Qué es exactamente una llamada a un LLM y cómo se estructura
- Los **4 conceptos fundamentales** que aparecen en todos los niveles posteriores: tokens, context window, system/user prompts y temperature
- Por qué este nivel es solo conceptual (no hay código ejecutable)

## ¿Qué es L1?

Una llamada a un modelo de lenguaje: le mandas texto, te devuelve texto.

No hay lógica, no hay herramientas, no hay memoria. Solo **una petición y una respuesta**. Es el bloque más pequeño con el que está construido todo lo demás — chains, agentes, RAG — son combinaciones y extensiones de esta operación básica.

## Anatomía de una petición

Cuando llamas a un LLM vía API mandas un objeto con estos campos:

| Campo | Descripción |
|-------|-------------|
| `model` | Qué modelo usar |
| `max_tokens` | Cuántos tokens puede generar como máximo en la respuesta |
| `temperature` | Cuánta aleatoriedad tiene la respuesta (`0` = determinista, `1` = creativo) |
| `system` | Instrucciones permanentes que definen el rol y el comportamiento |
| `messages` | El historial de la conversación: quién dijo qué |

**Ejemplo:**

```json
{
  "model": "claude-haiku-4-5-20251001",
  "max_tokens": 256,
  "temperature": 0,
  "system": "Eres un asistente de soporte técnico. Responde de forma concisa.",
  "messages": [
    { "role": "user", "content": "¿Qué significa un error 503?" }
  ]
}
```

## Anatomía de una respuesta

El modelo devuelve:

| Campo | Descripción |
|-------|-------------|
| `content` | Lista de bloques de contenido (normalmente uno de texto) |
| `stop_reason` | Por qué paró: `"end_turn"` (terminó), `"max_tokens"` (se cortó) |
| `usage` | Cuántos tokens consumió: `input_tokens` + `output_tokens` |

**Ejemplo:**

```json
{
  "content": [
    { "type": "text", "text": "Un 503 Service Unavailable significa que el servidor..." }
  ],
  "stop_reason": "end_turn",
  "usage": { "input_tokens": 32, "output_tokens": 41 }
}
```

## Conceptos clave

### 1. Tokens

El modelo no lee palabras — lee **tokens**. Un token es aproximadamente una sílaba o una palabra corta en inglés. En español suele ser algo más por palabra por la morfología del idioma.

```
"El servicio está caído"  →  6 tokens aproximadamente
"authentication"          →  3-4 tokens
```

Los tokens importan por dos razones:
1. Determinan el **coste** (se cobra por token de entrada y de salida)
2. Determinan el **límite** de lo que cabe en una conversación (el context window)

### 2. Context window

El espacio total que tiene el modelo para leer y generar. **Todo cuenta**: el system prompt, el historial de mensajes, y la respuesta que va a generar. Si la conversación supera ese límite, hay que truncar o resumir el historial.

Los modelos modernos tienen context windows de cientos de miles de tokens — suficiente para conversaciones largas o documentos completos.

### 3. System prompt vs User prompt

Son dos canales distintos con propósitos distintos:

|  | **System** | **User** |
|---|---|---|
| **Quién lo escribe** | El desarrollador | El usuario (o el código) |
| **Cuándo cambia** | Nunca (o raramente) | En cada mensaje |
| **Para qué sirve** | Definir el rol, las reglas, el formato de respuesta | El input concreto de cada interacción |

El **system prompt es el contrato permanente** con el modelo. El user prompt es lo que varía.

### 4. Temperature

Controla cómo de predecible es la respuesta del modelo al elegir la siguiente palabra.

| Valor | Comportamiento | Cuándo usarlo |
|-------|---------------|---------------|
| `0` | Siempre la respuesta más probable — determinista | Clasificación, extracción, parsing |
| `0.2 – 0.5` | Casi determinista, ligera variedad | Resúmenes, escritura técnica |
| `0.7+` | Creativo, variable | Brainstorming, generación de ideas |

**Regla práctica:** para tareas donde hay una respuesta correcta, usa `temperature=0`. Para tareas creativas, sube.

## ¿Por qué no hay código en L1?

Una llamada directa al modelo no requiere arquitectura — cualquier lenguaje con soporte HTTP puede hacerla. **El aprendizaje de L1 está en entender los conceptos**, no en escribir código.

Estos conceptos son los que luego explican:
- por qué L2 (chains) funciona como funciona
- por qué en L3 (tool use) hay un bucle
- por qué en L4 (RAG) metemos contexto en el prompt en lugar de esperar que el modelo "lo sepa"

Verás llamadas reales al modelo desde L2 en adelante.

## El flujo más simple posible

```
Developer define system prompt
        ↓
Usuario escribe un mensaje
        ↓
API recibe: model + system + messages
        ↓
Modelo genera tokens uno a uno hasta end_turn o max_tokens
        ↓
Respuesta llega como texto
```

## Siguientes pasos → L2

Una llamada aislada **no tiene memoria ni estado** — el modelo no recuerda lo que dijiste en el mensaje anterior a menos que lo incluyas en el historial.

El siguiente nivel, **[L2 — Chain determinista](../L2-CHAIN/L2_Chain_Determinista.ipynb)**, encadena varias llamadas pasando el output de una como input de la siguiente, con el código controlando el flujo.